# Umba Fraud Detection — EDA and Behavioural Signal Discovery

This notebook profiles the transaction and identity feeds, then translates the main fraud patterns into candidate behavioural features for modelling.

## Key Findings & EDA Summary

### Fraud Profile

Fraud accounts for approximately **3.4%** of all transactions, confirming a moderately imbalanced classification problem. The fraud population is sufficiently large to support a supervised learning baseline with class weighting.

### Transaction Behaviour

Transaction amount is highly right-skewed, with a small number of high-value transactions accounting for a disproportionate share of value. Fraud rates increase among larger transactions. The relationship is more useful when transaction amounts are evaluated relative to their operating context, especially country and channel medians.

### Sender and Recipient Behaviour

Recipient maturity is an important differentiator between fraudulent and legitimate transactions. Fraud rates are elevated among recipients with limited account history, while sender history shows a related but less pronounced pattern.

### Behavioural Intensity

Velocity-related variables show higher fraud incidence as activity levels increase. This suggests that activity intensity is a useful fraud signal, especially when combined with transaction value or weak identity evidence.

### Identity Behaviour

Identity consistency is one of the strongest observable fraud indicators. Fraud incidence increases as identity mismatches accumulate and identity information becomes less complete.

### Transaction Context

Fraud rates vary across channels and payment contexts. Channel-level behaviour is particularly important, with p2p and bank-transfer transactions showing materially higher fraud rates than the portfolio average.

### Temporal Behaviour

Fraud incidence varies through the observation period, which supports a validation design that evaluates the model on later transactions. This mirrors how the model would score incoming transactions in a live setting.

### Overall Conclusion

Fraud in this portfolio is best represented through four behavioural patterns: transaction deviation from expected norms, immature or lightly observed entities, weak identity footprints, and elevated activity levels. These observations form the basis for the feature engineering layer used in the modelling script.

### Implications for Feature Engineering

The modelling pipeline will convert these observations into compact behavioural features: amount deviation, sender and recipient maturity, identity confidence, behavioural intensity, and a small number of interaction features combining transaction value, identity risk, recipient risk, and activity intensity.


In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = 'data'  # update if running from another folder
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
identity = pd.read_csv(os.path.join(DATA_DIR, 'identity.csv'))

print('train', train.shape)
print('test', test.shape)
print('identity', identity.shape)
train.head()

## 1. Portfolio baseline

The labelled training set has 120,000 transactions and a fraud rate of **3.44%**. The test set has 40,000 transactions and starts strictly after the training period, matching the live scoring setup.

In [ ]:
baseline = {
    'train_rows': len(train),
    'test_rows': len(test),
    'fraud_count': int(train['isFraud'].sum()),
    'fraud_rate': train['isFraud'].mean(),
    'train_time_min': train['TransactionDT'].min(),
    'train_time_max': train['TransactionDT'].max(),
    'test_time_min': test['TransactionDT'].min(),
    'test_time_max': test['TransactionDT'].max(),
    'test_after_train': test['TransactionDT'].min() > train['TransactionDT'].max(),
}
pd.Series(baseline)

## 2. Transaction behaviour

Fraud is materially higher in **p2p** and **bank_transfer** channels. Country and currency do not separate fraud strongly by themselves, but amount must still be contextualised by country because KES and NGN use different numeric scales.

In [ ]:
channel_tbl = train.groupby('channel').agg(
    n=('isFraud', 'size'),
    fraud_rate=('isFraud', 'mean'),
    amount_median=('TransactionAmt', 'median')
).sort_values('fraud_rate', ascending=False)
channel_tbl

Amount is useful when expressed as a deviation from the relevant norm. The top decile of amount-vs-country-median has a fraud rate of about **6.98%**, roughly twice the portfolio baseline.

In [ ]:
eda = train.copy()
eda['amt_vs_country_median'] = eda['TransactionAmt'] / eda.groupby('country')['TransactionAmt'].transform('median')
eda['amt_vs_channel_median'] = eda['TransactionAmt'] / eda.groupby('channel')['TransactionAmt'].transform('median')
eda['amt_vs_country_bin'] = pd.qcut(eda['amt_vs_country_median'].rank(method='first'), 10, labels=False)
eda.groupby('amt_vs_country_bin').agg(
    n=('isFraud','size'),
    fraud_rate=('isFraud','mean'),
    ratio_min=('amt_vs_country_median','min'),
    ratio_median=('amt_vs_country_median','median'),
    ratio_max=('amt_vs_country_median','max')
)

## 3. Behavioural maturity

Recipient age is a strong operational signal. Transactions going to the youngest recipient accounts have a fraud rate of about **6.19%**. Sender history also shows useful lift: lower-history senders carry more risk than mature senders.

In [ ]:
for col in ['recipient_account_age_days', 'sender_prev_txn_count']:
    eda[f'{col}_bin'] = pd.qcut(eda[col].rank(method='first'), 10, labels=False)
    display(eda.groupby(f'{col}_bin').agg(
        n=('isFraud','size'),
        fraud_rate=('isFraud','mean'),
        value_min=(col,'min'),
        value_median=(col,'median'),
        value_max=(col,'max')
    ))

## 4. Identity match quality

The M-block is one of the clearest behavioural signal groups. More false identity matches correspond to higher fraud rates. Transactions with the highest false-match counts have a fraud rate of about **7.63%**.

In [ ]:
m_cols = [c for c in train.columns if c.startswith('M')]
eda['m_false_count'] = (eda[m_cols] == 'F').sum(axis=1)
eda['m_true_count'] = (eda[m_cols] == 'T').sum(axis=1)
eda['m_missing_count'] = eda[m_cols].isna().sum(axis=1)
eda['m_match_rate'] = eda['m_true_count'] / (eda['m_true_count'] + eda['m_false_count']).replace(0, np.nan)
eda['m_false_bin'] = pd.qcut(eda['m_false_count'].rank(method='first'), 10, labels=False)
eda.groupby('m_false_bin').agg(
    n=('isFraud','size'),
    fraud_rate=('isFraud','mean'),
    false_min=('m_false_count','min'),
    false_median=('m_false_count','median'),
    false_max=('m_false_count','max')
)

## 5. Velocity behaviour

The C-block behaves like velocity/count information. Aggregate velocity gives useful lift: the highest c_total decile has a fraud rate of about **4.81%** against the 3.44% baseline.

In [ ]:
c_cols = [c for c in train.columns if c.startswith('C') and c[1:].isdigit()]
eda['c_total'] = eda[c_cols].sum(axis=1)
eda['c_max'] = eda[c_cols].max(axis=1)
eda['c_nonzero_count'] = (eda[c_cols] > 0).sum(axis=1)
eda['c_total_bin'] = pd.qcut(eda['c_total'].rank(method='first'), 10, labels=False)
eda.groupby('c_total_bin').agg(
    n=('isFraud','size'),
    fraud_rate=('isFraud','mean'),
    c_total_min=('c_total','min'),
    c_total_median=('c_total','median'),
    c_total_max=('c_total','max')
)

## 6. Existing engineered variables

The strongest non-review single variables are V17, V3, V7 and V12. These should be kept in the model, while behavioural features add interpretability and operational meaning.

In [ ]:
from sklearn.metrics import average_precision_score
num_cols = train.select_dtypes(include=np.number).columns.drop(['isFraud', 'TransactionID'])
rows = []
for col in num_cols:
    x = train[col].fillna(train[col].median())
    ap_high = average_precision_score(train['isFraud'], x.rank(pct=True))
    ap_low = average_precision_score(train['isFraud'], (-x).rank(pct=True))
    rows.append({
        'feature': col,
        'missing_pct': train[col].isna().mean() * 100,
        'legit_median': train.loc[train.isFraud == 0, col].median(),
        'fraud_median': train.loc[train.isFraud == 1, col].median(),
        'best_univariate_ap': max(ap_high, ap_low),
        'direction': 'high' if ap_high >= ap_low else 'low'
    })
pd.DataFrame(rows).sort_values('best_univariate_ap', ascending=False).head(15)

## 7. Identity feed aggregation

The identity feed is session-level rather than transaction-level. We aggregate it to one row per transaction before modelling. Coverage is partial, so the presence or absence of identity data becomes a feature.

In [ ]:
identity_summary = {
    'identity_rows': len(identity),
    'unique_transaction_ids': identity['TransactionID'].nunique(),
    'extra_rows_from_multiple_sessions': len(identity) - identity['TransactionID'].nunique(),
    'train_transactions_with_identity': train['TransactionID'].isin(identity['TransactionID']).mean(),
    'test_transactions_with_identity': test['TransactionID'].isin(identity['TransactionID']).mean(),
}
pd.Series(identity_summary)

In [ ]:
id_num_cols = [c for c in identity.columns if c.startswith('id_')]
id_agg = identity.groupby('TransactionID').agg(
    identity_session_count=('TransactionID', 'size'),
    device_type_nunique=('DeviceType', lambda x: x.nunique(dropna=True)),
    device_info_nunique=('DeviceInfo', lambda x: x.nunique(dropna=True)),
    device_type_missing_rate=('DeviceType', lambda x: x.isna().mean()),
    device_info_missing_rate=('DeviceInfo', lambda x: x.isna().mean()),
).reset_index()

for col in id_num_cols:
    id_agg[f'{col}_mean'] = identity.groupby('TransactionID')[col].mean().values
    id_agg[f'{col}_missing_rate'] = identity.groupby('TransactionID')[col].apply(lambda x: x.isna().mean()).values

id_agg.head()

## 8. Review flag handling

`flagged_for_review` is highly predictive in train, but it is completely missing in test. For the model pipeline we will not use it as a normal predictive input. We can mention it in the write-up as a lifecycle field useful for feedback/labels, not live scoring.

In [ ]:
review_tbl = train.groupby('flagged_for_review').agg(
    n=('isFraud','size'),
    fraud_rate=('isFraud','mean')
)
print('test flagged_for_review missing rate:', test['flagged_for_review'].isna().mean())
review_tbl

## 9. Feature inventory for modelling

The v1 feature layer should include transaction context features, maturity features, velocity aggregates, identity quality features, and interactions. These signals are implementable in a training pipeline and can be served in a live system if the required historical aggregates are persisted.

In [ ]:
feature_inventory = pd.DataFrame([
    ['Transaction context', 'amt_vs_country_median, amt_vs_channel_median, country_channel, channel', 'Large value relative to local context'],
    ['Maturity', 'recipient_account_age_days, sender_prev_txn_count, low_sender_history, new_recipient', 'Newer / less mature entities are riskier'],
    ['Velocity', 'C1-C8, c_total, c_max, c_nonzero_count', 'Burst and repeated activity'],
    ['Identity quality', 'm_false_count, m_true_count, m_match_rate, m_missing_count', 'Identity mismatch and weak verification'],
    ['Identity sessions', 'identity_session_count, device_type_nunique, device_info_nunique, id_* aggregates', 'Session/device footprint'],
    ['Interactions', 'large_amount_x_new_recipient, high_velocity_x_weak_identity, large_amount_x_low_sender', 'Multiple moderate risks combining into stronger signal'],
    ['Existing engineered', 'V3, V7, V12, V17 and selected V features', 'Provided model signal with strong univariate lift'],
])
feature_inventory.columns = ['Feature group', 'Candidate features', 'Behaviour captured']
feature_inventory

## EDA conclusion

The strongest model direction is a behavioural XGBoost pipeline that excludes the review flag, contextualises amount by country/channel, preserves the strongest provided engineered variables, aggregates identity to transaction level, and adds interpretable maturity, velocity, identity-quality and interaction features. The fraud signal is not driven by one variable; it is built from combinations of transaction size, channel, recipient maturity, velocity and identity consistency.